In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Check folder structure
import os
for root, dirs, files in os.walk('/content/drive/MyDrive/'):
    for file in files:
        print(os.path.join(root, file))

/content/drive/MyDrive/Getting started.pdf
/content/drive/MyDrive/Ola income.gsheet
/content/drive/MyDrive/Copy of Income Budget NW FIRE Spreadsheet.gsheet
/content/drive/MyDrive/2024-05-31-Planned-Procurements.gsheet
/content/drive/MyDrive/Privacy Policy — Revix Digital Marketing Agency.gdoc
/content/drive/MyDrive/intro_to_modeling_fixed_tf2.ipynb
/content/drive/MyDrive/Documents/Onlinedrivingrecord.pdf
/content/drive/MyDrive/Documents/DL New_1.JPG
/content/drive/MyDrive/Documents/DL New_2.JPG
/content/drive/MyDrive/Documents/Immi-20180831-Email-801-Visa-Grant-Nomayer.pdf
/content/drive/MyDrive/Documents/Pass.jpg
/content/drive/MyDrive/Hustle/A. Heyward - Secret Millionaires Club.pdf
/content/drive/MyDrive/Hustle/Burke Hedges - The Parable of the Pipeline.pdf
/content/drive/MyDrive/Hustle/Copy of Jeff Walker - Launch.pdf
/content/drive/MyDrive/Hustle/Copy of Michel Fortin - Traffic Secrets.pdf
/content/drive/MyDrive/Hustle/Copy of MJ DeMarco - The Millionaire Fastlane.pdf
/content/dri

In [3]:
# Data path
data_path = '/content/drive/MyDrive/Colab Notebooks/ISY503-Assessment3-NLP/Data/domain_sentiment_data/sorted_data_acl/'

In [4]:
import os
import re
import pandas as pd

data_path = '/content/drive/MyDrive/Colab Notebooks/ISY503-Assessment3-NLP/Data/domain_sentiment_data/sorted_data_acl/'

categories = ['books', 'dvd', 'electronics', 'kitchen_&_housewares']

def parse_review_file(filepath, label):
    reviews = []
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()

    # Extract text between <review_text> tags
    matches = re.findall(r'<review_text>(.*?)</review_text>', content, re.DOTALL)

    for text in matches:
        text = text.strip()
        if text:
            reviews.append({'text': text, 'label': label})

    return reviews

all_reviews = []

for category in categories:
    pos_path = os.path.join(data_path, category, 'positive.review')
    neg_path = os.path.join(data_path, category, 'negative.review')

    if os.path.exists(pos_path):
        parsed = parse_review_file(pos_path, 1)
        all_reviews.extend(parsed)
        print(f"{category} - positive: {len(parsed)} reviews")

    if os.path.exists(neg_path):
        parsed = parse_review_file(neg_path, 0)
        all_reviews.extend(parsed)
        print(f"{category} - negative: {len(parsed)} reviews")

df = pd.DataFrame(all_reviews)

print(f"\nTotal reviews: {len(df)}")
print(f"Positive: {df['label'].sum()} | Negative: {(df['label'] == 0).sum()}")
df.head()

books - positive: 1000 reviews
books - negative: 1000 reviews
dvd - positive: 1000 reviews
dvd - negative: 1000 reviews
electronics - positive: 1000 reviews
electronics - negative: 1000 reviews
kitchen_&_housewares - positive: 1000 reviews
kitchen_&_housewares - negative: 1000 reviews

Total reviews: 8000
Positive: 4000 | Negative: 4000


,text,label
0,Sphere by Michael Crichton is an excellant nov...,1
1,Dr. Oz is an accomplished heart surgeon in the...,1
2,The most gorgeous artwork in comic books. Cont...,1
3,This book is for lovers of Robicheaux. His de...,1
4,This is going to be a short and sweet review b...,1


In [5]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Download nltk corpus
nltk.download('all')

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# Cleaning the data
def clean_text(text):
    # Lowercase
    text = text.lower()
    # Remove HTML tags (some reviews have leftovers)
    text = re.sub(r'<.*?>', '', text)
    # Remove punctuation and numbers
    text = re.sub(r'[^a-z\s]', '', text)
    # Tokenise the text
    tokens = word_tokenize(text)
    # Remove stopwords and lemmatize
    tokens = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words]
    return ' '.join(tokens)

df['cleaned_text'] = df['text'].apply(clean_text)

print("Cleaning done.")
print(f"\nOriginal: {df['text'][0][:200]}")
print(f"\nCleaned:  {df['cleaned_text'][0][:200]}")

[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/abc.zip.
[nltk_data]    | Downloading package alpino to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/alpino.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_eng to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping
[nltk_data]    |       taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_ru to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping
[nltk_data]    |       taggers/averaged_perceptron_tagger_ru.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_rus to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |  

Cleaning done.

Original: Sphere by Michael Crichton is an excellant novel. This was certainly the hardest to put down of all of the Crichton novels that I have read. 

The story revolves around a man named Norman Johnson. Joh

Cleaned:  sphere michael crichton excellant novel certainly hardest put crichton novel read story revolves around man named norman johnson johnson phycologist travel civilans remote location pacific ocean help 


In [6]:
# Check word count distribution
df['word_count'] = df['cleaned_text'].apply(lambda x: len(x.split()))

print(f"Min words: {df['word_count'].min()}")
print(f"Max words: {df['word_count'].max()}")
print(f"Mean words: {df['word_count'].mean():.0f}")

# Remove outlier reviews - too short (under 5 word) or too long (over 500)
df = df[(df['word_count'] >= 5) & (df['word_count'] <= 500)]

print(f"\nAfter outlier removal: {len(df)} reviews")
print(f"Positive: {df['label'].sum()} | Negative: {(df['label'] == 0).sum()}")

Min words: 1
Max words: 1808
Mean words: 69

After outlier removal: 7951 reviews
Positive: 3974 | Negative: 3977


In [7]:
# Mix and randomise
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print("Data shuffled.")
df.head()

Data shuffled.


,text,label,cleaned_text,word_count
0,Maybe it is me but this novel was about as exc...,0,maybe novel exciting watching grass grow first...,85
1,This collection of the first four Road picture...,1,collection first four road picture well worth ...,26
2,I was really looking forward to receiving this...,0,really looking forward receiving four week pur...,18
3,The first book of Auster's New York trilogy wa...,0,first book austers new york trilogy originally...,231
4,You must have received CA-709 (without the cas...,1,must received ca without cassette component mi...,11


In [8]:
from collections import Counter

# Build vocabulary from all cleaned reviews
all_words = ' '.join(df['cleaned_text']).split()
word_counts = Counter(all_words)

# Sort by frequency, most common first
vocab = sorted(word_counts, key=word_counts.get, reverse=True)

# Create word-to-integer mapping (start from 1, reserve 0 for padding)
word_to_int = {word: idx for idx, word in enumerate(vocab, 1)}

print(f"Vocabulary size: {len(word_to_int)}")
print(f"\nSample mappings:")
for word in list(word_to_int.items())[:10]:
    print(f"  '{word[0]}' → {word[1]}")

Vocabulary size: 39024

Sample mappings:
  'book' → 1
  'one' → 2
  'like' → 3
  'time' → 4
  'movie' → 5
  'would' → 6
  'get' → 7
  'good' → 8
  'great' → 9
  'work' → 10


In [9]:
import numpy as np

# Encode each review as a list of integers
def encode_reviews(df, word_to_int):
    encoded = []
    for review in df['cleaned_text']:
        encoded.append([word_to_int.get(word, 0) for word in review.split()])
    return encoded

encoded_reviews = encode_reviews(df, word_to_int)
labels = df['label'].values

print(f"Sample encoded review (first 10 tokens):")
print(encoded_reviews[0][:10])
print(f"\nCorresponding label: {labels[0]}")

Sample encoded review (first 10 tokens):
[211, 145, 1339, 279, 7747, 1814, 19, 599, 145, 20]

Corresponding label: 0


In [10]:
def pad_truncate(encoded_reviews, seq_length=200):
    features = np.zeros((len(encoded_reviews), seq_length), dtype=int)

    for i, review in enumerate(encoded_reviews):
        if len(review) <= seq_length:
            # Pad with zeros at the beginning
            features[i, -len(review):] = review
        else:
            # Truncate to first seq_length tokens
            features[i, :] = review[:seq_length]

    return features

seq_length = 200
features = pad_truncate(encoded_reviews, seq_length)

print(f"Features shape: {features.shape}")
print(f"\nSample padded review (first 20 values):")
print(features[0][:20])

Features shape: (7951, 200)

Sample padded review (first 20 values):
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [11]:
split_frac = 0.8

train_end = int(len(features) * split_frac)
val_end = int(len(features) * 0.9)

train_x = features[:train_end]
val_x = features[train_end:val_end]
test_x = features[val_end:]

train_y = labels[:train_end]
val_y = labels[train_end:val_end]
test_y = labels[val_end:]

print(f"Training set:   {train_x.shape}")
print(f"Validation set: {val_x.shape}")
print(f"Test set:       {test_x.shape}")

Training set:   (6360, 200)
Validation set: (795, 200)
Test set:       (796, 200)


In [12]:
import torch
from torch.utils.data import DataLoader, TensorDataset

# Convert to tensors
train_data = TensorDataset(
    torch.from_numpy(train_x).long(),
    torch.from_numpy(train_y).float()
)
val_data = TensorDataset(
    torch.from_numpy(val_x).long(),
    torch.from_numpy(val_y).float()
)
test_data = TensorDataset(
    torch.from_numpy(test_x).long(),
    torch.from_numpy(test_y).float()
)

batch_size = 50

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_data,   batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_data,  batch_size=batch_size, shuffle=False)

print(f"Train batches:      {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches:       {len(test_loader)}")

Train batches:      128
Validation batches: 16
Test batches:       16


In [13]:
import torch.nn as nn

class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, output_size, embedding_dim,
                 hidden_dim, n_layers, drop_prob=0.5):
        super(SentimentLSTM, self).__init__()

        self.hidden_dim = hidden_dim
        self.n_layers   = n_layers

        # Layers
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm      = nn.LSTM(embedding_dim, hidden_dim, n_layers,
                                 dropout=drop_prob, batch_first=True)
        self.dropout   = nn.Dropout(drop_prob)
        self.fc        = nn.Linear(hidden_dim, output_size)
        self.sigmoid   = nn.Sigmoid()

    def forward(self, x, hidden):
        batch_size = x.size(0)
        embeds     = self.embedding(x)
        lstm_out, hidden = self.lstm(embeds, hidden)
        lstm_out   = lstm_out.contiguous().view(-1, self.hidden_dim)
        out        = self.dropout(lstm_out)
        out        = self.fc(out)
        out        = self.sigmoid(out)
        out        = out.view(batch_size, -1)
        out        = out[:, -1]
        return out, hidden

    def init_hidden(self, batch_size):
        device = next(self.parameters()).device
        h0 = torch.zeros(self.n_layers, batch_size, self.hidden_dim).to(device)
        c0 = torch.zeros(self.n_layers, batch_size, self.hidden_dim).to(device)
        return (h0, c0)

# Instantiate the model
vocab_size    = len(word_to_int) + 1  # +1 for padding token
output_size   = 1
embedding_dim = 400
hidden_dim    = 256
n_layers      = 2

model = SentimentLSTM(vocab_size, output_size, embedding_dim, hidden_dim, n_layers)

# Use GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = model.to(device)

print(model)
print(f"\nTraining on: {device}")

SentimentLSTM(
  (embedding): Embedding(39025, 400)
  (lstm): LSTM(400, 256, num_layers=2, batch_first=True, dropout=0.5)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=256, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)

Training on: cuda


In [14]:
from torch import optim

class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, output_size, embedding_dim,
                 hidden_dim, n_layers, drop_prob=0.5):
        super(SentimentLSTM, self).__init__()
        self.hidden_dim   = hidden_dim
        self.n_layers     = n_layers
        self.embedding    = nn.Embedding(vocab_size, embedding_dim)
        self.lstm         = nn.LSTM(embedding_dim, hidden_dim, n_layers,
                                    dropout=drop_prob,
                                    batch_first=True,
                                    bidirectional=True)   # ← key change
        self.dropout      = nn.Dropout(drop_prob)
        self.fc           = nn.Linear(hidden_dim * 2, output_size)  # *2 for bidirectional
        self.sigmoid      = nn.Sigmoid()

    def forward(self, x, hidden):
        batch_size        = x.size(0)
        embeds            = self.embedding(x)
        lstm_out, hidden  = self.lstm(embeds, hidden)
        lstm_out          = lstm_out.contiguous().view(-1, self.hidden_dim * 2)
        out               = self.dropout(lstm_out)
        out               = self.fc(out)
        out               = self.sigmoid(out)
        out               = out.view(batch_size, -1)
        out               = out[:, -1]
        return out, hidden

    def init_hidden(self, batch_size):
        device = next(self.parameters()).device
        # *2 for bidirectional
        h0     = torch.zeros(self.n_layers * 2, batch_size, self.hidden_dim).to(device)
        c0     = torch.zeros(self.n_layers * 2, batch_size, self.hidden_dim).to(device)
        return (h0, c0)

# Hyperparameters
vocab_size    = len(word_to_int) + 1
output_size   = 1
embedding_dim = 200
hidden_dim    = 128
n_layers      = 2     # back to 2 so dropout warning goes away

model     = SentimentLSTM(vocab_size, output_size, embedding_dim, hidden_dim, n_layers)
model     = model.to(device)

epochs    = 15
lr        = 0.0003
clip      = 5
patience  = 4

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

best_val_loss     = float('inf')
epochs_no_improve = 0
best_model_state  = None

model.train()

for epoch in range(epochs):
    train_losses = []

    for inputs, labels in train_loader:
        actual_batch   = inputs.size(0)
        h              = model.init_hidden(actual_batch)
        inputs, labels = inputs.to(device), labels.to(device)
        h              = tuple([each.data for each in h])

        model.zero_grad()
        output, h      = model(inputs, h)
        loss           = criterion(output.squeeze(), labels.float())
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        train_losses.append(loss.item())

    # Validation
    val_losses  = []
    val_correct = 0
    val_total   = 0

    model.eval()
    with torch.no_grad():
        for inputs, labels in val_loader:
            actual_batch   = inputs.size(0)
            val_h          = model.init_hidden(actual_batch)
            inputs, labels = inputs.to(device), labels.to(device)
            val_h          = tuple([each.data for each in val_h])

            output, val_h  = model(inputs, val_h)
            val_loss       = criterion(output.squeeze(), labels.float())
            val_losses.append(val_loss.item())

            predictions    = torch.round(output.squeeze())
            val_correct   += (predictions == labels).sum().item()
            val_total     += labels.size(0)

    avg_val_loss = sum(val_losses) / len(val_losses)
    val_accuracy = val_correct / val_total * 100

    print(f"Epoch {epoch+1}/{epochs} | "
          f"Train Loss: {sum(train_losses)/len(train_losses):.4f} | "
          f"Val Loss: {avg_val_loss:.4f} | "
          f"Val Accuracy: {val_accuracy:.2f}%")

    if avg_val_loss < best_val_loss:
        best_val_loss      = avg_val_loss
        best_model_state   = model.state_dict().copy()
        epochs_no_improve  = 0
        print(f"  ✓ Best model saved (val loss: {best_val_loss:.4f})")
    else:
        epochs_no_improve += 1
        print(f"  No improvement ({epochs_no_improve}/{patience})")
        if epochs_no_improve >= patience:
            print(f"\nEarly stopping triggered at epoch {epoch+1}")
            break

    model.train()

model.load_state_dict(best_model_state)
print("\nBest model restored for testing.")

Epoch 1/15 | Train Loss: 0.6842 | Val Loss: 0.6586 | Val Accuracy: 61.64%
  ✓ Best model saved (val loss: 0.6586)
Epoch 2/15 | Train Loss: 0.5920 | Val Loss: 0.5899 | Val Accuracy: 67.55%
  ✓ Best model saved (val loss: 0.5899)
Epoch 3/15 | Train Loss: 0.4851 | Val Loss: 0.5436 | Val Accuracy: 72.83%
  ✓ Best model saved (val loss: 0.5436)
Epoch 4/15 | Train Loss: 0.3913 | Val Loss: 0.5342 | Val Accuracy: 73.96%
  ✓ Best model saved (val loss: 0.5342)
Epoch 5/15 | Train Loss: 0.3181 | Val Loss: 0.5192 | Val Accuracy: 76.48%
  ✓ Best model saved (val loss: 0.5192)
Epoch 6/15 | Train Loss: 0.2429 | Val Loss: 0.5433 | Val Accuracy: 77.61%
  No improvement (1/4)
Epoch 7/15 | Train Loss: 0.1908 | Val Loss: 0.6396 | Val Accuracy: 76.48%
  No improvement (2/4)
Epoch 8/15 | Train Loss: 0.1483 | Val Loss: 0.7507 | Val Accuracy: 76.86%
  No improvement (3/4)
Epoch 9/15 | Train Loss: 0.1082 | Val Loss: 0.7884 | Val Accuracy: 77.61%
  No improvement (4/4)

Early stopping triggered at epoch 9

Best

In [16]:
test_losses  = []
test_correct = 0
test_total   = 0

model.eval()
with torch.no_grad():
    for inputs, labels in test_loader:
        actual_batch   = inputs.size(0)
        test_h         = model.init_hidden(actual_batch)
        inputs, labels = inputs.to(device), labels.to(device)
        test_h         = tuple([each.data for each in test_h])

        output, test_h = model(inputs, test_h)
        loss           = criterion(output.squeeze(), labels.float())
        test_losses.append(loss.item())

        predictions    = torch.round(output.squeeze())
        test_correct  += (predictions == labels).sum().item()
        test_total    += labels.size(0)

test_accuracy = test_correct / test_total * 100
print(f"Test Loss:     {sum(test_losses)/len(test_losses):.4f}")
print(f"Test Accuracy: {test_accuracy:.2f}%")

Test Loss:     0.7924
Test Accuracy: 76.88%


In [18]:
def predict_sentiment(review_text, model, word_to_int, seq_length=200):
    model.eval()
    cleaned      = clean_text(review_text)
    encoded      = [word_to_int.get(word, 0) for word in cleaned.split()]

    if len(encoded) < seq_length:
        encoded  = [0] * (seq_length - len(encoded)) + encoded
    else:
        encoded  = encoded[:seq_length]

    input_tensor = torch.tensor([encoded], dtype=torch.long).to(device)
    h            = model.init_hidden(1)
    h            = tuple([each.data for each in h])

    with torch.no_grad():
        output, h = model(input_tensor, h)

    prob  = output.item()
    label = "Positive review" if prob >= 0.5 else "Negative review"
    print(f"Prediction : {label}")
    print(f"Confidence : {prob:.4f}")
    return label

# Test both directions
predict_sentiment("This product is absolutely amazing, I love it!", model, word_to_int)
predict_sentiment("Terrible quality, broke after one day. Complete waste of money.", model, word_to_int)

Prediction : Positive review
Confidence : 0.9678
Prediction : Negative review
Confidence : 0.0031


'Negative review'

In [28]:
import os

save_dir  = '/content/drive/MyDrive/Colab Notebooks/ISY503-Assessment3-NLP/models/'
os.makedirs(save_dir, exist_ok=True)

torch.save({
    'model_state_dict': model.state_dict(),
    'word_to_int':      word_to_int,
    'vocab_size':       vocab_size,
    'embedding_dim':    embedding_dim,
    'hidden_dim':       hidden_dim,
    'n_layers':         n_layers,
}, save_dir + 'lstm_model.pt')

print("LSTM model saved.")

LSTM model saved.
